In [20]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [21]:
df = pd.read_csv(r"archive\CombinedClean.csv")

In [22]:
df = df.drop(["Recalled Word"], axis=1) 

In [23]:
word_sim_df = pd.read_csv("word_similarity_matrix.csv")
word_sim_matrix = word_sim_df.to_numpy()
word_vec_df = pd.read_csv("word_vec_fasttext.csv")
word_vec_matrix = word_vec_df.T.to_numpy()

In [24]:
df.columns

Index(['ListID', 'Presented Word', 'Present Position', 'Recall Position',
       'Reaction Time', 'Hit', 'Participant ID', 'Participant Group'],
      dtype='object')

In [25]:
df = df.drop(df[df["ListID"]==0].index, axis=0) #get rid of the training set

In [26]:
df = df.sort_values(by=["Participant Group","Participant ID", "ListID","Present Position"]) #sort
df = df.reset_index()

In [27]:
"""##Distance to Most Similar Item
#compare within lists
#use similarity matrix constructed 
last_list_id = df["ListID"][0]
last_participant_id = df["Participant ID"][0]
last_participant_group = df["Participant Group"][0]

for row in range(df.shape[0]): #iterating through rows
    # MOST SIMILAR VAL
    # MOST SIMILAR PRES DISTANCE
    while df["Participant ID"][row] == last_participant_id and df[""]"""

'##Distance to Most Similar Item\n#compare within lists\n#use similarity matrix constructed \nlast_list_id = df["ListID"][0]\nlast_participant_id = df["Participant ID"][0]\nlast_participant_group = df["Participant Group"][0]\n\nfor row in range(df.shape[0]): #iterating through rows\n    # MOST SIMILAR VAL\n    # MOST SIMILAR PRES DISTANCE\n    while df["Participant ID"][row] == last_participant_id and df[""]'

In [28]:
word_sim_df.set_index('Unnamed: 0', inplace=True)
word_sim_df.index.name = None

### Position Zone

In [29]:
df["POSITION_ZONE"] = pd.cut(
    df["Present Position"],
    bins = [0,3,14,18],
    labels=[1,2,3]
).astype(int) #divide positions to zones

### Extract Similarity

In [30]:
def find_most_similiar(group, sim_df): 
    length = len(group)-1
    similarities = []
    distances = []
    pos = []
    words = group["Presented Word"].values #get rid of pandas indexing
    
    i = 0
    while i <= length: 
        most_sim_index = i 
        current_word = words[i]
        most_sim_val = -2

        j = 0
        while j < i: 
            comparison_word = words[j]
            try: 
                similarity = sim_df.loc[current_word, comparison_word]
                if pd.notna(similarity) and similarity > most_sim_val:
                    most_sim_val, most_sim_index = similarity, j
            except KeyError: 
                pass
            j += 1

        if most_sim_val < -1: 
            similarities.append(np.nan)
            distances.append(np.nan)
            pos.append(np.nan)
        else:
            similarities.append(most_sim_val)
            distances.append(i-most_sim_index)
            pos.append(most_sim_index)
        i += 1
    return pd.DataFrame(data = {"MOST_SIM_VAL": similarities,
                             "MOST_SIM_DIST": distances, "DEBUG_MOST_SIM_POS": pos}, index=group.index)

In [31]:
simliarity_features_vd = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(find_most_similiar, word_sim_df)
clean_features = simliarity_features_vd.droplevel([0,1,2])
df = df.join(clean_features)

C:\Users\elito\AppData\Local\Temp\ipykernel_12920\2313958469.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  simliarity_features_vd = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(find_most_similiar, word_sim_df)


In [32]:
df

,index,ListID,Presented Word,Present Position,Recall Position,Reaction Time,Hit,Participant ID,Participant Group,POSITION_ZONE,MOST_SIM_VAL,MOST_SIM_DIST,DEBUG_MOST_SIM_POS
0,3231,1,SARI,1,3.0,0.000000,1,0,eng,1,NaN,NaN,NaN
1,3232,1,KAĞIT,2,2.0,0.000000,1,0,eng,1,0.381701,1.0,0.0
2,3233,1,YAZI,3,8.0,0.000000,1,0,eng,1,0.385806,1.0,1.0
3,3234,1,EKRAN,4,NaN,NaN,0,0,eng,2,0.252005,1.0,2.0
4,3235,1,PARK,5,NaN,NaN,0,0,eng,2,0.314455,3.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3031,1495,4,KALIP,13,NaN,NaN,0,20,soc,2,0.473795,1.0,11.0
3032,1496,4,HARİTA,14,4.0,2.799603,1,20,soc,2,0.384075,12.0,1.0
3033,1497,4,SULUK,15,3.0,3.816858,1,20,soc,3,NaN,NaN,NaN
3034,1498,4,PARA,16,2.0,2.069633,1,20,soc,3,0.428927,5.0,10.0


### Hit - Miss Feature

In [33]:
#hit or miss of previous trial 
def prev_hit_miss(group): 
    length = len(group)-1
    hit_miss = []
    hit_list = group["Hit"].values #get rid of pandas indexing
    
    i = 0
    while i <= length: 
        cur = i 
        cur_val = hit_list[i]
        if i > 0:
            hit_miss.append(hit_list[i-1])
        else:
            hit_miss.append(np.nan)
        i+=1
    return pd.DataFrame(data = {"PREV_HIT_MISS": hit_miss}, index=group.index)

In [34]:
hit_miss_feature = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(prev_hit_miss)
hit_miss_feature = hit_miss_feature.droplevel([0,1,2])
df = df.join(hit_miss_feature)
df.head()

C:\Users\elito\AppData\Local\Temp\ipykernel_12920\3132261422.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hit_miss_feature = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(prev_hit_miss)


,index,ListID,Presented Word,Present Position,Recall Position,Reaction Time,Hit,Participant ID,Participant Group,POSITION_ZONE,MOST_SIM_VAL,MOST_SIM_DIST,DEBUG_MOST_SIM_POS,PREV_HIT_MISS
0,3231,1,SARI,1,3.0,0.0,1,0,eng,1,NaN,NaN,NaN,NaN
1,3232,1,KAĞIT,2,2.0,0.0,1,0,eng,1,0.381701,1.0,0.0,1.0
2,3233,1,YAZI,3,8.0,0.0,1,0,eng,1,0.385806,1.0,1.0,1.0
3,3234,1,EKRAN,4,NaN,NaN,0,0,eng,2,0.252005,1.0,2.0,1.0
4,3235,1,PARK,5,NaN,NaN,0,0,eng,2,0.314455,3.0,1.0,0.0


### AVG Similarity With Previous n Words

In [35]:
def avg_sim_previous(group, sim_df, n): 
    length = len(group)
    avg_similarities = []
    words = group["Presented Word"].values 

    i = 0
    while i < length:
        current_word = words[i]
        
        # THE FIX: This determines the start of your look-back window.
        # If i=5 and n=3, start_idx is 2. (It checks j=2, 3, 4)
        # If i=1 and n=3, start_idx is 0. (It checks j=0)
        start_idx = max(0, i - n)
        
        sim_sum = 0
        valid_count = 0
        
        # Loop ONLY from the start_idx up to the current word
        j = start_idx
        while j < i:
            comparison_word = words[j]
            try:
                similarity = sim_df.loc[current_word, comparison_word]
                
                # If the similarity exists, add it to our running total
                if pd.notna(similarity):
                    sim_sum += similarity
                    valid_count += 1
            except KeyError:
                pass
            j += 1
            
        # Calculate the average. If no valid words were found (or i=0), return NaN.
        if valid_count > 0:
            avg_similarities.append(sim_sum / valid_count)
        else:
            avg_similarities.append(np.nan)
            
        i += 1
        
    # Return a DataFrame with a dynamic column name based on 'n'
    return pd.DataFrame(
        data={f"AVG_SIM_PREV_{n}": avg_similarities}, 
        index=group.index
    )

In [36]:
avg_similarity_n = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(avg_sim_previous, word_sim_df, 3)
avg_similarity_n = avg_similarity_n.droplevel([0,1,2])
df = df.join(avg_similarity_n)
df.head()

C:\Users\elito\AppData\Local\Temp\ipykernel_12920\3642068541.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  avg_similarity_n = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(avg_sim_previous, word_sim_df, 3)


,index,ListID,Presented Word,Present Position,Recall Position,Reaction Time,Hit,Participant ID,Participant Group,POSITION_ZONE,MOST_SIM_VAL,MOST_SIM_DIST,DEBUG_MOST_SIM_POS,PREV_HIT_MISS,AVG_SIM_PREV_3
0,3231,1,SARI,1,3.0,0.0,1,0,eng,1,NaN,NaN,NaN,NaN,NaN
1,3232,1,KAĞIT,2,2.0,0.0,1,0,eng,1,0.381701,1.0,0.0,1.0,0.381701
2,3233,1,YAZI,3,8.0,0.0,1,0,eng,1,0.385806,1.0,1.0,1.0,0.325940
3,3234,1,EKRAN,4,NaN,NaN,0,0,eng,2,0.252005,1.0,2.0,1.0,0.227335
4,3235,1,PARK,5,NaN,NaN,0,0,eng,2,0.314455,3.0,1.0,0.0,0.272572


### Similarity to Last Word

In [39]:
def sim_last_word(group, sim_df): 
    length = len(group)-1
    similarities = []
    words = group["Presented Word"].values #get rid of pandas indexing
    
    i = 0
    while i <= length:
        current_word = words[i]
        if i>0: 
            comparison_word = words[i-1]
            try: 
                similarity = sim_df.loc[current_word, comparison_word]
                similarities.append(similarity)
            except KeyError: 
                similarities.append(np.nan)
        else: 
            similarities.append(np.nan)
        i += 1
    return pd.DataFrame(data = {"PREV_SIM": similarities}, index = group.index)

In [40]:
prev_word_sim = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(sim_last_word, word_sim_df)
prev_word_sim = prev_word_sim.droplevel([0,1,2])
df = df.join(prev_word_sim)
df.head()

C:\Users\elito\AppData\Local\Temp\ipykernel_12920\2403950683.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prev_word_sim = df.groupby(["Participant ID", "Participant Group", "ListID"]).apply(sim_last_word, word_sim_df)


,index,ListID,Presented Word,Present Position,Recall Position,Reaction Time,Hit,Participant ID,Participant Group,POSITION_ZONE,MOST_SIM_VAL,MOST_SIM_DIST,DEBUG_MOST_SIM_POS,PREV_HIT_MISS,AVG_SIM_PREV_3,PREV_SIM
0,3231,1,SARI,1,3.0,0.0,1,0,eng,1,NaN,NaN,NaN,NaN,NaN,NaN
1,3232,1,KAĞIT,2,2.0,0.0,1,0,eng,1,0.381701,1.0,0.0,1.0,0.381701,0.381701
2,3233,1,YAZI,3,8.0,0.0,1,0,eng,1,0.385806,1.0,1.0,1.0,0.325940,0.385806
3,3234,1,EKRAN,4,NaN,NaN,0,0,eng,2,0.252005,1.0,2.0,1.0,0.227335,0.252005
4,3235,1,PARK,5,NaN,NaN,0,0,eng,2,0.314455,3.0,1.0,0.0,0.272572,0.222346


### NEXT: CONTRAST
How much words semantic content differs from all the rest. 